In [1]:
import pandas as pd

In [2]:
# Import the CSV, convert both date columns to datetime, and create two new columns: Total_Before_Discount = Quantity * Unit_Price, Total_After_Discount = Total_Before_Discount * (1 - Discount_Pct / 100), Shipping_Days = number of days between Order_Date and Ship_Date
df = pd.read_csv('ecommerce_orders.csv')
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Ship_Date'] = pd.to_datetime(df['Ship_Date'])
df['Total_Before_Discount'] = (df['Quantity'] * df['Unit_Price'])
df['Total_After_Discount'] = (df['Total_Before_Discount'] * (1 - df['Discount_Pct'] / 100)).round(2)
df['Shipping_Days'] = (df['Ship_Date'] - df['Order_Date']).dt.days
df

,Order_ID,Customer_ID,Order_Date,Ship_Date,Product,Category,Quantity,Unit_Price,Discount_Pct,Region,Payment_Method,Total_Before_Discount,Total_After_Discount,Shipping_Days
0,O001,C201,2024-01-02,2024-01-04,Laptop,Electronics,1,999.99,0,East,Credit Card,999.99,999.99,2
1,O002,C202,2024-01-03,2024-01-07,Headphones,Electronics,2,79.99,10,West,PayPal,159.98,143.98,4
2,O003,C203,2024-01-05,2024-01-06,Desk Chair,Furniture,1,349.99,0,East,Credit Card,349.99,349.99,1
3,O004,C201,2024-01-08,2024-01-09,Mouse,Electronics,3,29.99,5,East,Credit Card,89.97,85.47,1
4,O005,C204,2024-01-10,2024-01-15,Bookshelf,Furniture,1,199.99,15,South,Debit Card,199.99,169.99,5
5,O006,C205,2024-01-12,2024-01-13,Keyboard,Electronics,2,59.99,0,West,PayPal,119.98,119.98,1
6,O007,C202,2024-01-15,2024-01-22,Sofa,Furniture,1,899.99,20,West,Credit Card,899.99,719.99,7
7,O008,C206,2024-01-17,2024-01-18,Tablet,Electronics,1,499.99,10,North,Debit Card,499.99,449.99,1
8,O009,C203,2024-01-20,2024-01-21,Lamp,Furniture,4,44.99,0,East,Credit Card,179.96,179.96,1
9,O010,C207,2024-01-22,2024-01-28,Monitor,Electronics,1,349.99,5,North,PayPal,349.99,332.49,6


In [3]:
# Create a pivot table showing the sum of Total_After_Discount by Region (rows) and Category (columns). Add row totals and column totals (margins). Round to 2 decimal places.
ptable = pd.pivot_table(df, values = 'Total_After_Discount', index = 'Region', columns = 'Category', aggfunc = 'sum', fill_value=0, margins = True).round(2)
ptable

Category,Electronics,Furniture,All
Region,,,
East,1464.40,1229.93,2694.33
North,782.48,764.97,1547.45
South,949.99,709.98,1659.97
West,398.91,919.98,1318.89
All,3595.78,3624.86,7220.64


In [4]:
# For each Customer_ID, calculate their total spend (sum of Total_After_Discount), number of orders, and average Shipping_Days. Then create a column called Customer_Tier using the following logic: Total spend > 1000 → 'Gold', Total spend between 500 and 1000 inclusive → 'Silver', Total spend below 500 → 'Bronze', Display sorted by total spend descending.
result = df.groupby('Customer_ID').agg(Total_Spend = ('Total_After_Discount', 'sum'), Total_Orders = ('Order_ID', 'size'), Avg_Shipping_Days = ('Shipping_Days', 'mean')).reset_index()
result['Customer_Tier'] = (result['Total_Spend'].apply(lambda x: 'Gold' if x > 1000 else ('Silver' if x >= 500 and x <= 1000 else 'Bronze')))
result = result.sort_values('Total_Spend', ascending = False)
result

,Customer_ID,Total_Spend,Total_Orders,Avg_Shipping_Days,Customer_Tier
0,C201,1293.43,4,1.25,Gold
3,C204,1119.98,2,3.50,Gold
1,C202,998.92,3,4.00,Silver
2,C203,700.92,3,1.00,Silver
8,C209,699.98,1,1.00,Silver
9,C210,674.99,1,6.00,Silver
7,C208,539.99,1,4.00,Silver
5,C206,539.97,2,1.00,Silver
6,C207,332.49,1,6.00,Bronze
4,C205,319.97,2,3.50,Bronze


In [5]:
# Find the top 3 products by total revenue (Total_After_Discount). Then find what percentage of total overall revenue each of those top 3 products represents. Display with product name, total revenue, and percentage.
topthreeproducts = df.groupby('Product')['Total_After_Discount'].sum().reset_index(name = 'Product_Revenue')
total_amount = topthreeproducts['Product_Revenue'].sum()
topthreeproducts['Percentage'] = ((topthreeproducts['Product_Revenue'] / total_amount) * 100).round(2)
topthreeproducts = topthreeproducts.nlargest(3, 'Product_Revenue')
topthreeproducts

,Product,Product_Revenue,Percentage
6,Laptop,1949.98,27.01
9,Sofa,1394.98,19.32
1,Desk Chair,1049.97,14.54


In [6]:
# Create a column called Month extracted from Order_Date. Then calculate the month-over-month revenue growth rate for the entire business (not per customer). Display the growth rate as a percentage rounded to 1 decimal place.
df['Month'] = (df['Order_Date'].dt.month)
newchart = df.groupby('Month')['Total_After_Discount'].sum().reset_index(name = 'Monthly_Revenue')
newchart['MoM_Change'] = (((newchart['Monthly_Revenue'] - newchart['Monthly_Revenue'].shift(1)) / newchart['Monthly_Revenue'].shift(1)) * 100).round(2)
newchart

,Month,Monthly_Revenue,MoM_Change
0,1,4231.80,NaN
1,2,2988.84,-29.37
